# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rishipatel092005/Fly-Rank-/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method Choice and Why

My lane is Refresh / Content Opportunity Scoring, framed as a ranking problem.

For the modeling stage, I will predict the observed declining outcome for each content page and use the model's predicted probability of decline as the ranking score.

I will compare three methods:

- Logistic Regression — a simple and interpretable model that provides a useful linear baseline.
- Decision Tree — captures non-linear relationships while remaining relatively easy to interpret.
- Random Forest — combines multiple decision trees and can capture more complex relationships between search and content signals.

I will choose the model based on measured performance against the Week-4 baseline rather than assuming that a more complex model is automatically better.

The model is intended for directional decision support. A high predicted probability of decline does not prove that refreshing the page will cause better future performance.

## 2. Split Design

I will use a client-level holdout rather than randomly splitting individual pages.

A client can contain multiple content pages, so putting pages from the same client into both training and test sets could make the evaluation unrealistically easy.

I will therefore hold out complete clients for testing. The split will be reproducible using a fixed random seed.

The Week-4 baseline and all ML models will be evaluated on the same held-out clients and using the same ranking metric, so the comparison is fair.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [39]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print(
    "Declining rate:",
    round(df["is_declining_label"].mean(), 3)
)
RANDOM_STATE = 42

clients = (
    df["client_id"]
    .fillna("unknown")
    .astype(str)
    .unique()
)

rng = np.random.default_rng(RANDOM_STATE)
rng.shuffle(clients)

test_client_count = max(
    1,
    int(round(len(clients) * 0.20))
)

test_clients = set(
    clients[:test_client_count]
)

test_mask = (
    df["client_id"]
    .fillna("unknown")
    .astype(str)
    .isin(test_clients)
)

train_idx = np.where(~test_mask)[0]
test_idx = np.where(test_mask)[0]

print("Train rows:", len(train_idx))
print("Test rows:", len(test_idx))

train_clients = set(
    df.iloc[train_idx]["client_id"].dropna().astype(str)
)

test_clients_actual = set(
    df.iloc[test_idx]["client_id"].dropna().astype(str)
)

print(
    "Client overlap:",
    len(train_clients.intersection(test_clients_actual))
)

Rows: 30000
Columns: 44
Declining rate: 0.542
Train rows: 27675
Test rows: 2325
Client overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 3. Train + Compare vs My Baseline

I will train multiple supervised models to estimate the probability that a content page is observed as declining.

The models will use only information available at the decision time and will exclude label-derived or future-outcome fields.

I will compare Logistic Regression, Decision Tree, and Random Forest.

For each model, the predicted probability of decline will be used as the ranking score. I will evaluate the models and the Week-4 baseline on the same held-out clients using the same Precision@50 metric.

The goal is not to assume that ML is better, but to measure whether any model improves ranking quality over the simple Week-4 baseline.

In [40]:
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

categorical_features = [
    "content_type",
    "main_intent",
    "competition_level",
    "age_tier",
    "freshness_tier",
    "position_tier",
]

X_num = (
    df[numeric_features]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

X_cat = (
    df[categorical_features]
    .fillna("unknown")
    .astype(str)
)

X_cat = pd.get_dummies(
    X_cat,
    columns=categorical_features,
    dtype=float
)

X = pd.concat(
    [
        X_num.reset_index(drop=True),
        X_cat.reset_index(drop=True)
    ],
    axis=1
)

y = df["is_declining_label"]

print("Feature matrix shape:", X.shape)

Feature matrix shape: (30000, 41)


In [41]:
X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (27675, 41)
X_test: (2325, 41)


In [42]:
def precision_at_k(y_true, scores, k=50):
    temp = pd.DataFrame({
        "y_true": np.asarray(y_true),
        "score": np.asarray(scores)
    })

    top_k = temp.sort_values(
        "score",
        ascending=False
    ).head(k)

    return float(top_k["y_true"].mean())

In [61]:
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

work = df.copy()

numeric_cols = [
    "search_volume",
    "ctr",
    "avg_position"
]

for col in numeric_cols:
    work[col] = pd.to_numeric(work[col], errors="coerce")
    work[col] = work[col].fillna(work[col].median())

work["volume_score"] = work["search_volume"].rank(pct=True)
work["ctr_opportunity"] = 1 - work["ctr"].rank(pct=True)
work["position_opportunity"] = work["avg_position"].rank(pct=True)

work["baseline_score"] = (
    0.40 * work["volume_score"]
    + 0.30 * work["ctr_opportunity"]
    + 0.30 * work["position_opportunity"]
)

def get_reason(row):
    signals = {
        "HIGH_VOLUME_OPPORTUNITY": row["volume_score"],
        "CTR_OPPORTUNITY": row["ctr_opportunity"],
        "POSITION_OPPORTUNITY": row["position_opportunity"]
    }
    return max(signals, key=signals.get)

work["reason_code"] = work.apply(get_reason, axis=1)

work["action"] = np.where(
    work["baseline_score"] >= work["baseline_score"].quantile(0.80),
    "REVIEW",
    "MONITOR"
)

work = work.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

work["rank"] = np.arange(1, len(work) + 1)

queue = work[
    [
        "rank",
        "content_id",
        "baseline_score",
        "reason_code",
        "action"
    ]
].copy()

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

queue.to_csv(output_path, index=False)

print("Rows ranked:", len(queue))
print("Output written to:", output_path)

Rows ranked: 30000
Output written to: work/outputs/baseline_action_score.csv


In [62]:
baseline = pd.read_csv(
    "work/outputs/baseline_action_score.csv"
)

baseline_lookup = baseline.set_index(
    "content_id"
)["baseline_score"]

print("Baseline rows:", len(baseline))

Baseline rows: 30000


In [65]:
test_content_ids = df.iloc[test_idx]["content_id"]

baseline_test_scores = (
    test_content_ids
    .map(baseline_lookup)
    .fillna(0)
    .to_numpy()
)

baseline_p50 = precision_at_k(
    y_test,
    baseline_test_scores,
    k=50
)

print(
    "Week-4 Baseline Precision@50:",
    round(baseline_p50, 3)
)

Week-4 Baseline Precision@50: 0.46


In [66]:
results = []

for name, model in models.items():

    # Train
    model.fit(X_train, y_train)

    # Predicted probability of decline
    probabilities = model.predict_proba(X_test)[:, 1]

    # Ranking metric
    p50 = precision_at_k(
        y_test,
        probabilities,
        k=50
    )

    results.append({
        "Method": name,
        "Precision@50": p50
    })

# Add Week-4 baseline
baseline_row = pd.DataFrame([{
    "Method": "Week-4 Baseline",
    "Precision@50": baseline_p50
}])

comparison = pd.concat(
    [
        baseline_row,
        pd.DataFrame(results)
    ],
    ignore_index=True
)

comparison = comparison.sort_values(
    "Precision@50",
    ascending=False
).reset_index(drop=True)

display(comparison)

,Method,Precision@50
0,Logistic Regression,0.58
1,Decision Tree,0.58
2,Week-4 Baseline,0.46
3,Random Forest,0.42


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [68]:
lr_model = models["Logistic Regression"]
dt_model = models["Decision Tree"]

lr_model.fit(X_train, y_train)
dt_model.fit(X_train, y_train)

lr_prob = lr_model.predict_proba(X_test)[:, 1]
dt_prob = dt_model.predict_proba(X_test)[:, 1]

In [69]:
error_frame = df.iloc[test_idx][
    ["content_id", "ctr", "avg_position"]
].copy()

error_frame["actual_declining"] = y_test.to_numpy()
error_frame["lr_probability"] = lr_prob
error_frame["dt_probability"] = dt_prob

display(error_frame.head(10))

,content_id,ctr,avg_position,actual_declining,lr_probability,dt_probability
11,content_5a3e876cf7f7,0.00,0.0,0,0.121866,0.013965
48,content_326fa2fa449f,0.00,8.3,1,0.709902,0.408491
61,content_d99c66ea5462,11.11,2.0,0,0.067855,0.480397
63,content_d5d3c2e98937,0.00,28.9,1,0.603106,0.480397
69,content_95d488a56079,0.89,12.1,0,0.559422,0.609342
71,content_ced59bb3a1a6,0.00,2.7,0,0.114595,0.210690
88,content_998f6f88784c,0.02,2.6,1,0.290240,0.716553
91,content_48724397d104,5.26,2.5,1,0.287490,0.480397
108,content_6e02c8d0cc61,0.14,17.5,1,0.671050,0.716553
116,content_5883af3441ba,15.38,10.2,0,0.113066,0.480397


In [70]:
lr_false_positives = error_frame[
    (error_frame["lr_probability"] >= 0.5) &
    (error_frame["actual_declining"] == 0)
]

print("Logistic Regression false positives:", len(lr_false_positives))

display(lr_false_positives.head(10))

Logistic Regression false positives: 480


,content_id,ctr,avg_position,actual_declining,lr_probability,dt_probability
69,content_95d488a56079,0.89,12.1,0,0.559422,0.609342
147,content_bec800684271,1.79,27.4,0,0.553364,0.715537
168,content_d7cbd76b788d,0.11,6.4,0,0.592155,0.716553
198,content_c3e86d4031b6,0.00,10.2,0,0.654554,0.716553
251,content_7dff534db3ae,0.31,5.3,0,0.574474,0.419691
278,content_f7de2c48d051,0.00,14.0,0,0.603351,0.210690
339,content_f4e177f6d346,0.18,29.0,0,0.519965,0.609342
465,content_29102284b855,0.29,21.6,0,0.589434,0.609342
494,content_218ca439f951,0.00,40.4,0,0.524298,0.716553
577,content_5920115cc1ad,0.00,44.3,0,0.519149,0.716553


In [71]:
lr_false_negatives = error_frame[
    (error_frame["lr_probability"] < 0.5) &
    (error_frame["actual_declining"] == 1)
]

print("Logistic Regression false negatives:", len(lr_false_negatives))

display(lr_false_negatives.head(10))

Logistic Regression false negatives: 333


,content_id,ctr,avg_position,actual_declining,lr_probability,dt_probability
88,content_998f6f88784c,0.02,2.6,1,0.290240,0.716553
91,content_48724397d104,5.26,2.5,1,0.287490,0.480397
275,content_ea851c8c0ad2,0.00,2.4,1,0.321881,0.480397
279,content_421272323961,16.67,9.3,1,0.158867,0.480397
292,content_b5367e6afc79,0.28,12.3,1,0.000160,0.609342
318,content_d2e655334ee2,12.50,2.6,1,0.061124,0.632959
424,content_4148b1c5a86e,0.00,11.3,1,0.354101,0.503572
477,content_aaee0ce51abf,0.00,7.5,1,0.462697,0.110173
580,content_b70b7c263bd2,0.00,66.8,1,0.388908,0.716553
601,content_ee1b0d51721b,0.00,2.4,1,0.325706,0.480397


In [72]:
tree_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": dt_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(tree_importance.head(15))

,feature,importance
5,impressions_90d,0.411219
9,content_age_days,0.251830
12,avg_position,0.094615
6,clicks_90d,0.087478
14,scroll_rate,0.056859
7,pageviews_90d,0.032084
4,char_count,0.030405
11,ctr,0.027956
10,days_since_last_update,0.007402
1,competition,0.000091


In [73]:
lr_coefficients = pd.DataFrame({
    "feature": X.columns,
    "coefficient": lr_model.named_steps["model"].coef_[0]
})

lr_coefficients["absolute_effect"] = (
    lr_coefficients["coefficient"].abs()
)

lr_coefficients = lr_coefficients.sort_values(
    "absolute_effect",
    ascending=False
)

display(lr_coefficients.head(15))

,feature,coefficient,absolute_effect
3,word_count,1.493124,1.493124
4,char_count,-1.394367,1.394367
7,pageviews_90d,-0.348165,0.348165
40,position_tier_top_3,-0.334597,0.334597
12,avg_position,-0.268555,0.268555
8,sessions_90d,0.216106,0.216106
9,content_age_days,-0.203667,0.203667
11,ctr,-0.175427,0.175427
23,main_intent_unknown,0.156220,0.156220
17,content_type_feedly article,-0.139575,0.139575


## 4. Errors and Interpretation

I reviewed model errors rather than relying only on the Precision@50 score.

Logistic Regression and Decision Tree achieved the highest measured Precision@50 of 0.58, compared with 0.46 for the Week-4 baseline. Random Forest achieved 0.42 and did not outperform the baseline.

I will inspect false positives and false negatives to understand where the models make mistakes. I will also inspect model coefficients or feature importance to understand which observed signals contribute most to the predictions.

These findings are observational and directional. A model prediction does not prove that a content refresh will improve future performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [ ]:
import os
print(os.getcwd())